### 1. create_sql_query_chian

In [1]:
#例1.
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
import dotenv
import os

dotenv.load_dotenv()
#测试连接本地MYSQL数据库
db_host = "localhost"
db_port = "1433"
db_database = "day20"
driver = "ODBC Driver 17 for SQL Server"

#mssql+pymysql://用户名:密码@ip地址:端口号/数据库名?driver=
db = SQLDatabase.from_uri(
    f"mssql+pyodbc://@{db_host}:{db_port}/{db_database}?driver={driver}",
)

print("操作的是哪种数据库:", db.dialect)
print("获取数据库中的表:", db.get_usable_table_names())

#执行查询操作
res = db.run("SELECT COUNT(*) FROM Comment")
print(res)
# model = ChatTongyi(
#     api_key=os.getenv("DASHSCOPE_API_KEY"),
#     model="qwen-flash", 
#     temperature=0)
# chain = create_sql_query_chain(model, db)
# response = chain.invoke({"question": "How many employees are there"})

操作的是哪种数据库: mssql
获取数据库中的表: ['Comment', 'libraryuser', 'novel_categories', 'novel_info', 'novel_info_backup']
[(5011,)]


In [2]:
#例2 chain 的使用
# pip install -U langchain langchain-community langchain-openai
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
import dotenv
import os

dotenv.load_dotenv()
#1.连接本地MYSQL数据库
db_host = "localhost"
db_port = "1433"
db_database = "day20"
driver = "ODBC Driver 17 for SQL Server"

#mssql+pymysql://用户名:密码@ip地址:端口号/数据库名?driver=
db = SQLDatabase.from_uri(
    f"mssql+pyodbc://@{db_host}:{db_port}/{db_database}?driver={driver}",
)
#2.获取大模型
model = ChatTongyi(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model="qwen-flash", 
    temperature=0)

#3.创建create_sql_query_chain的示例
chain = create_sql_query_chain(model, db)#专门用于将自然语言转换为 SQL 查询的链
response = chain.invoke(
    {"question": "Comment表中一共有多少条评论"},
    {"table_name_to_use":["Comment"]}                    
    )
print(response)

SQLQuery: SELECT COUNT(*) AS TotalComments FROM [Comment]


### 2. create_stuff_documents_chian

In [3]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.documents import Document

# 定义提示词模板
prompt = PromptTemplate.from_template("""
如下文档{docs}中说，香蕉是什么颜色的？
""")

# 创建链
llm = ChatTongyi(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model="qwen-flash",)
chain = create_stuff_documents_chain(
    llm, 
    prompt,
    document_variable_name="docs"#与提示词中的{docs}对应
)

# 文档输入
docs = [
    Document(
        page_content="苹果，学名Malus pumila Mill.，别称西洋苹果、柰，属于蔷薇科苹果属的植物。苹果是全球最广泛种植和销售的水果之一，具有悠久的栽培历史和广泛的分布范围。苹果的原始种群主要起源于中亚的天山山脉附近，尤其是现代哈萨克斯坦的阿拉木图地区，提供了所有现代苹果品种的基因库。苹果通过早期的贸易路线，如丝绸之路，从中亚向外扩散到全球各地。"
    ),
    Document(
        page_content="香蕉是白色的水果，主要产自热带地区。"
    ),
    Document(
        page_content="蓝莓是蓝色的浆果，含有抗氧化物质。"
    )
]

chain.invoke({"docs": docs})

'根据您提供的文档内容，文中提到：“香蕉是白色的水果”，但这是错误的描述。\n\n实际上，香蕉成熟时通常是**黄色**的，而不是白色。虽然香蕉的果肉在切开后呈现白色或淡白色，但整体外观（尤其是外皮）在成熟时为黄色。因此，正确的说法是：**香蕉是黄色的水果**，主要产自热带地区。\n\n所以，回答您的问题：  \n文中说“香蕉是白色的水果”——这不符合事实。  \n正确答案是：**香蕉是黄色的**。'